# HypNO-ARZ vs Exact Riemann Solver
Load a checkpoint and interactively compare model output against the exact Riemann solution for any IC you choose.

In [1]:
%matplotlib inline
import sys
sys.path.insert(0, '..')   # repo root

import numpy as np
import matplotlib.pyplot as plt
import torch

from hyperbolic_pde.arz import physics_arz as P
from hyperbolic_pde.arz.riemann_arz import solve_riemann_arz_xt
from hyperbolic_pde.arz.model_arz import load_hypno_arz_from_checkpoint

P.set_pressure_form('rho')
print('pressure_form =', P.get_pressure_form())

pressure_form = rho


In [2]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT = r'C:\Users\dimit\Desktop\repos\DL-for-HPDE\hyperbolic_pde\runs\hypno_arz\run_20260608_111316\checkpoint_epoch160.pt'
DEVICE = 'cpu'  # GTX 1650 Ti is 4GB — use cpu for safe inference

# Grid (must match training)
NX   = 128
NT   = 128
XMIN = -1.0
XMAX =  1.0
TMAX =  1.0

# Hi-res grid for exact solution plots
NX_HI = 1000
# ──────────────────────────────────────────────────────────────────────────────

model, _ = load_hypno_arz_from_checkpoint(CKPT, device=DEVICE)
model.eval()
print(f'Loaded checkpoint: {CKPT}')
print(f'Device: {DEVICE}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

dx    = (XMAX - XMIN) / NX
x_mid = np.array([XMIN + (i + 0.5) * dx for i in range(NX)], dtype=np.float32)
t_grid = np.linspace(0.0, TMAX, NT, dtype=np.float32)
x_hi   = np.linspace(XMIN, XMAX, NX_HI, dtype=np.float64)

x_t = torch.tensor(x_mid,  dtype=torch.float32, device=DEVICE)
t_t = torch.tensor(t_grid, dtype=torch.float32, device=DEVICE)

[HypNO_ARZ] kx=6 kt=4 d_latent=64 d_hidden=64 layers=8 skip=False normalize_edge_offsets=True use_relaxation_features=False double_batch=False
Loaded checkpoint: C:\Users\dimit\Desktop\repos\DL-for-HPDE\hyperbolic_pde\runs\hypno_arz\run_20260608_111316\checkpoint_epoch160.pt
Device: cpu
Parameters: 474,085


In [ ]:
def make_riemann_ic(rhoL, uL, rhoR, uR, x0=0.0):
    """Build cell-midpoint IC arrays (rho0, w0) for a Riemann problem."""
    wL = uL + P.pressure(rhoL)
    wR = uR + P.pressure(rhoR)
    rho0 = np.where(x_mid <= x0, rhoL, rhoR).astype(np.float32)
    w0   = np.where(x_mid <= x0, wL,   wR  ).astype(np.float32)
    return rho0, w0


def run_model(rho0, w0):
    """Run HypNO-ARZ on (rho0, w0) IC arrays. Returns (rho, w, v) of shape (NT, NX)."""
    with torch.no_grad():
        rho_p, w_p, _ = model(
            torch.tensor(rho0[None], dtype=torch.float32, device=DEVICE),
            torch.tensor(w0[None],   dtype=torch.float32, device=DEVICE),
            x_t, t_t,
        )
    rho_p = rho_p[0].cpu().numpy()
    w_p   = w_p[0].cpu().numpy()
    v_p   = w_p - P.pressure(rho_p)
    return rho_p, w_p, v_p


def run_exact(rhoL, uL, rhoR, uR, x0=0.0):
    """Exact Riemann solution on the hi-res grid. Returns (rho, w, v) of shape (NT, NX_HI)."""
    wL = uL + P.pressure(rhoL)
    wR = uR + P.pressure(rhoR)
    rho_e, w_e, v_e = solve_riemann_arz_xt(
        rhoL, wL, rhoR, wR,
        x_hi, t_grid.astype(np.float64), x0=x0,
    )
    return rho_e.astype(np.float32), w_e.astype(np.float32), v_e.astype(np.float32)


def plot_comparison(rhoL, uL, rhoR, uR, x0=0.0, title=''):
    """
    Side-by-side: exact (hi-res) vs model (cell midpoints).
    Rows: rho, w, v.  Columns: exact | model | |error| (on model grid).
    """
    rho0, w0 = make_riemann_ic(rhoL, uL, rhoR, uR, x0)
    rho_m, w_m, v_m = run_model(rho0, w0)
    rho_e, w_e, v_e = run_exact(rhoL, uL, rhoR, uR, x0)

    # Exact sampled at midpoints for error computation
    rho_e_mid, w_e_mid, _ = solve_riemann_arz_xt(
        rhoL, float(uL + P.pressure(rhoL)),
        rhoR, float(uR + P.pressure(rhoR)),
        x_mid.astype(np.float64), t_grid.astype(np.float64), x0=x0,
    )
    v_e_mid = w_e_mid - P.pressure(rho_e_mid)

    mae_rho = float(np.mean(np.abs(rho_m - rho_e_mid)))
    mae_w   = float(np.mean(np.abs(w_m   - w_e_mid)))
    mae_v   = float(np.mean(np.abs(v_m   - v_e_mid)))

    channels = [
        (r'$\rho$', rho_e, rho_m, rho_e_mid, 'viridis'),
        (r'$w$',    w_e,   w_m,   w_e_mid,   'magma'),
        (r'$v$',    v_e,   v_m,   v_e_mid,   'cividis'),
    ]
    maes = [mae_rho, mae_w, mae_v]

    fig, axes = plt.subplots(3, 3, figsize=(15, 10), constrained_layout=True)
    for row, (name, exact_hi, model_field, exact_mid, cmap) in enumerate(channels):
        vmin = min(float(exact_hi.min()), float(model_field.min()))
        vmax = max(float(exact_hi.max()), float(model_field.max()))

        # Exact (hi-res)
        im = axes[row, 0].pcolormesh(x_hi, t_grid, exact_hi, cmap=cmap,
                                      vmin=vmin, vmax=vmax, shading='nearest')
        axes[row, 0].set_title(f'Exact {name}')
        fig.colorbar(im, ax=axes[row, 0])

        # Model (cell midpoints)
        im = axes[row, 1].pcolormesh(x_mid, t_grid, model_field, cmap=cmap,
                                      vmin=vmin, vmax=vmax, shading='nearest')
        axes[row, 1].set_title(f'Model {name}  (MAE={maes[row]:.3e})')
        fig.colorbar(im, ax=axes[row, 1])

        # |error| on model grid
        err = np.abs(model_field - exact_mid)
        im = axes[row, 2].pcolormesh(x_mid, t_grid, err, cmap='magma',
                                      vmin=0, shading='nearest')
        axes[row, 2].set_title(f'|error| {name}')
        fig.colorbar(im, ax=axes[row, 2])

        for ax in axes[row]:
            ax.set_xlabel('x'); ax.set_ylabel('t')

    suptitle = (
        f'{title}\n'
        f'L: (ρ={rhoL}, u={uL})   R: (ρ={rhoR}, u={uR})   x0={x0}\n'
        f'MAE  ρ={mae_rho:.3e}  w={mae_w:.3e}  v={mae_v:.3e}'
    )
    fig.suptitle(suptitle, fontsize=11)
    plt.show()
    return rho_m, w_m, v_m, rho_e, w_e, v_e

: 

## Case 1: 1-shock + 2-contact
`rhoL < rhoM`  →  density increases across the 1-wave  →  shock.

In [ ]:
plot_comparison(rhoL=0.3, uL=0.4, rhoR=0.7, uR=0.6, x0=0.0,
                title='Case 1: shock + contact');

## Case 2: 1-rarefaction + 2-contact
`rhoL > rhoM`  →  density decreases across the 1-wave  →  rarefaction fan.

In [ ]:
plot_comparison(rhoL=0.8, uL=0.1, rhoR=0.2, uR=0.5, x0=0.0,
                title='Case 2: rarefaction + contact');

## Case 3: contact-only (degenerate 1-wave)
`uL == uR`  →  no 1-wave, only the 2-contact moves.

In [ ]:
plot_comparison(rhoL=0.2, uL=0.3, rhoR=0.7, uR=0.3, x0=0.0,
                title='Case 3: contact only');

## Case 4: off-centre interface

In [ ]:
plot_comparison(rhoL=0.4, uL=0.2, rhoR=0.6, uR=0.5, x0=-0.3,
                title='Case 4: shock + contact, x0=-0.3');

## Freeplay — edit and run

In [ ]:
plot_comparison(
    rhoL = 0.5,
    uL   = 0.2,
    rhoR = 0.5,
    uR   = 0.6,
    x0   = 0.0,
    title = 'Custom case',
);

## Time slices — model vs exact at selected t values

In [ ]:
def plot_slices(rhoL, uL, rhoR, uR, x0=0.0, t_vals=(0.25, 0.5, 0.75, 1.0), title=''):
    rho0, w0 = make_riemann_ic(rhoL, uL, rhoR, uR, x0)
    rho_m, w_m, v_m = run_model(rho0, w0)
    rho_e, w_e, v_e = run_exact(rhoL, uL, rhoR, uR, x0)

    fig, axes = plt.subplots(len(t_vals), 2, figsize=(12, 3.5 * len(t_vals)),
                              constrained_layout=True)
    if len(t_vals) == 1:
        axes = axes[None]

    for row, tv in enumerate(t_vals):
        k = int(np.argmin(np.abs(t_grid - tv)))
        t_actual = float(t_grid[k])

        # rho
        axes[row, 0].plot(x_hi,  rho_e[k], 'k-',  lw=1.5, label='exact')
        axes[row, 0].plot(x_mid, rho_m[k], 'r--', lw=1.2, label='model')
        axes[row, 0].set_title(f't={t_actual:.2f}  ρ')
        axes[row, 0].set_xlabel('x'); axes[row, 0].legend(); axes[row, 0].grid(alpha=0.3)

        # v
        axes[row, 1].plot(x_hi,  v_e[k], 'k-',  lw=1.5, label='exact')
        axes[row, 1].plot(x_mid, v_m[k], 'b--', lw=1.2, label='model')
        axes[row, 1].set_title(f't={t_actual:.2f}  v')
        axes[row, 1].set_xlabel('x'); axes[row, 1].legend(); axes[row, 1].grid(alpha=0.3)

    fig.suptitle(title or f'L=({rhoL},{uL})  R=({rhoR},{uR})  x0={x0}')
    plt.show()


plot_slices(rhoL=0.3, uL=0.4, rhoR=0.7, uR=0.6, title='Shock + contact — slices')
plot_slices(rhoL=0.8, uL=0.1, rhoR=0.2, uR=0.5, title='Rarefaction + contact — slices')